# 00 — Model Smoke Test

**목적**: Colab T4에서 `Qwen/Qwen2.5-7B-Instruct` 4bit NF4 로드 가능 여부 확인 + VRAM 측정  
**조건**: GPU 런타임 필수 (런타임 → 런타임 유형 변경 → T4 GPU)

In [1]:
# 필요 패키지 설치 (Colab 기준)
!pip install -q transformers==4.44.2 bitsandbytes==0.43.3 accelerate==0.33.0 peft==0.12.0

ERROR: Could not find a version that satisfies the requirement bitsandbytes==0.43.3 (from versions: 0.31.8, 0.32.0, 0.32.1, 0.32.2, 0.32.3, 0.33.0, 0.33.1, 0.34.0, 0.35.0, 0.35.1, 0.35.2, 0.35.3, 0.35.4, 0.36.0, 0.36.0.post1, 0.36.0.post2, 0.37.0, 0.37.1, 0.37.2, 0.38.0, 0.38.0.post1, 0.38.0.post2, 0.38.1, 0.39.0, 0.39.1, 0.40.0, 0.40.0.post1, 0.40.0.post2, 0.40.0.post3, 0.40.0.post4, 0.40.1, 0.40.1.post1, 0.40.2, 0.41.0, 0.41.1, 0.41.2, 0.41.2.post1, 0.41.2.post2, 0.41.3, 0.41.3.post1, 0.41.3.post2, 0.42.0, 0.49.0, 0.49.1, 0.49.2)
ERROR: No matching distribution found for bitsandbytes==0.43.3


In [ ]:
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

SEED = 42
torch.manual_seed(SEED)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def vram_usage_gb() -> float:
    """현재 GPU VRAM 사용량(GB) 반환."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / (1024 ** 3)

def vram_reserved_gb() -> float:
    """PyTorch가 예약한 전체 VRAM(GB) 반환 (allocated + fragmentation)."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_reserved() / (1024 ** 3)

def gpu_total_gb() -> float:
    """GPU 전체 메모리(GB) 반환."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM     : {gpu_total_gb():.2f} GB")

In [ ]:
# ── 4bit NF4 양자화 설정 (CLAUDE.md 고정 사양) ──────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,   # QLoRA 권장 설정
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("[1/2] 토크나이저 로드 중 ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

vram_before = vram_reserved_gb()
t0 = time.time()

print("[2/2] 모델 로드 중 (최초 실행 시 ~14GB 다운로드) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

load_sec = time.time() - t0
vram_after = vram_reserved_gb()

print(f"\n로드 완료")
print(f"  소요 시간    : {load_sec:.1f}s")
print(f"  VRAM before  : {vram_before:.2f} GB")
print(f"  VRAM after   : {vram_after:.2f} GB")
print(f"  VRAM delta   : {vram_after - vram_before:.2f} GB")
print(f"  Total GPU    : {gpu_total_gb():.2f} GB")
print(f"  여유 VRAM    : {gpu_total_gb() - vram_after:.2f} GB")

In [ ]:
# ── 추론 테스트 ──────────────────────────────────────────────────────────────
SYSTEM_PROMPT = "충남대학교 학내 정보 안내 도우미입니다."
USER_INPUT = "안녕"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": USER_INPUT},
]

input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

t1 = time.time()
with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=128,
        do_sample=False,           # greedy (재현성)
        temperature=None,
        top_p=None,
    )
gen_sec = time.time() - t1

generated = output_ids[0][input_ids.shape[-1]:]
response = tokenizer.decode(generated, skip_special_tokens=True)

new_tokens = generated.shape[0]
print(f"입력  : {USER_INPUT}")
print(f"출력  : {response}")
print(f"\n생성 토큰 수 : {new_tokens}")
print(f"생성 시간    : {gen_sec:.2f}s  ({new_tokens / gen_sec:.1f} tok/s)")
print(f"추론 후 VRAM : {vram_reserved_gb():.2f} GB")

In [ ]:
# ── 결과 요약 ────────────────────────────────────────────────────────────────
print("=" * 50)
print("SMOKE TEST 결과")
print("=" * 50)
ok_load   = vram_after > 0
ok_answer = len(response.strip()) > 0
ok_vram   = vram_after < 14.5   # T4 15GB 기준 여유 확보

print(f"[{'PASS' if ok_load   else 'FAIL'}] 모델 로드")
print(f"[{'PASS' if ok_answer else 'FAIL'}] 응답 생성")
print(f"[{'PASS' if ok_vram   else 'FAIL'}] VRAM < 14.5 GB  (실측: {vram_after:.2f} GB)")

if all([ok_load, ok_answer, ok_vram]):
    print("\n모든 항목 PASS — T4에서 QLoRA 학습/추론 진행 가능")
else:
    print("\n일부 항목 FAIL — VRAM 초과 시 max_memory 또는 batch size 조정 필요")